In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle
import rdkit.Chem as Chem
from rdkit.Chem import AllChem,DataStructs
from sklearn.metrics.pairwise import cosine_similarity
import gseapy as gp
import model


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Load the pre-trained model
model = model.DrugEncoder()
model.load_state_dict(torch.load('moable.pth'))
model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
model.eval()

# Function to convert SMILES string to fingerprint
def smiles2fp(smilesstr):
    mol = Chem.MolFromSmiles(smilesstr)
    fp_obj = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048, useChirality=True)
    arr = np.zeros((1,))
    DataStructs.ConvertToNumpyArray(fp_obj, arr)
    return arr

# Function to generate embeddings for the drug
def drug_embeddings(drug_dict):
    embeddings = {}
    for drug, smiles in drug_dict.items():
        fingerprint = smiles2fp(smiles)
        fingerprint_tensor = torch.tensor(fingerprint).unsqueeze(0).float().to(device)
        with torch.no_grad():
            embedding = model(fingerprint_tensor).cpu().numpy().flatten()
        embeddings[drug] = embedding
    return embeddings

# Create a dictionary with your compound
drug_dict = {
    "kavain": "COC1=CC(=O)OC(C1)/C=C/c2ccccc2"
}


In [2]:
# Generate embeddings for the drug
CP_embedding_dict = drug_embeddings(drug_dict)

[12:46:04] DEPRECATION WARNING: please use MorganGenerator


In [3]:
# Load gene perturbation signature and embedding dictionaries
GP_sig_df = pd.read_pickle('data/GP_sig_df.pkl')
GP_embedding_dict = pd.read_pickle('data/GP_embedding_dict.zip')


In [4]:
# Calculate cosine similarity between drug and gene perturbation embeddings
similarities = cosine_similarity(list(CP_embedding_dict.values()), list(GP_embedding_dict.values()))


In [6]:
print(similarities)

[[ 0.16659346  0.2776516   0.01100294 ... -0.01568493 -0.20123859
   0.35962927]]


In [7]:
# Process the results for nifurtimox
os.makedirs("data/output", exist_ok=True)
for i, drug in enumerate(CP_embedding_dict):
    connectivity_score_df = pd.DataFrame({
        'sig_id': list(GP_embedding_dict.keys()), 
        'score': similarities[i]
    })
    connectivity_score_df = connectivity_score_df.merge(GP_sig_df[['sig_id', 'cmap_name']])
    connectivity_score_df = connectivity_score_df.groupby(['cmap_name']).max().sort_values(by=['score'], ascending=False).reset_index()
    
    # Perform gene set enrichment analysis
    pathway_res = gp.prerank(rnk=connectivity_score_df[['cmap_name', 'score']], gene_sets='KEGG_2016', min_size=5, max_size=300, permutation_num=100)
    
    # Save results to CSV
    output_path = f"data/output/{drug}_pathway_results.csv"
    pathway_res.res2d.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")
    
    # Display top pathways
    print(f"Top pathways for {drug}:")
    print(pathway_res.res2d.head())

2024-08-13 12:47:47,988 [WARNING] Duplicated values found in preranked stats: 0.03% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top pathways for kavain:
      Name                                               Term        ES  \
0  prerank        p53 signaling pathway Homo sapiens hsa04115  0.487384   
1  prerank          MicroRNAs in cancer Homo sapiens hsa05206  0.394512   
2  prerank     Chronic myeloid leukemia Homo sapiens hsa05220  0.443141   
3  prerank       Acute myeloid leukemia Homo sapiens hsa05221   0.43466   
4  prerank  Neurotrophin signaling pathway Homo sapiens hs...   0.40285   

        NES NOM p-val FDR q-val FWER p-val   Tag %  Gene %  \
0  2.459125       0.0       0.0        0.0   43/55  36.17%   
1  2.337797       0.0       0.0        0.0  82/129  34.68%   
2  2.333661       0.0       0.0        0.0   51/70  38.29%   
3  2.268057       0.0       0.0        0.0   42/57  38.29%   
4  2.252756       0.0       0.0        0.0  76/105  40.77%   

                                          Lead_genes  
0  CCND2;TP53;CASP3;DDB2;FAS;PTEN;CYCS;PMAIP1;ZMA...  
1  CDC25B;CCND2;TP53;CASP3;EZH2;PTEN;CDKN

In [8]:
 print(pathway_res.res2d.head(20))

       Name                                               Term        ES  \
0   prerank        p53 signaling pathway Homo sapiens hsa04115  0.487384   
1   prerank          MicroRNAs in cancer Homo sapiens hsa05206  0.394512   
2   prerank     Chronic myeloid leukemia Homo sapiens hsa05220  0.443141   
3   prerank       Acute myeloid leukemia Homo sapiens hsa05221   0.43466   
4   prerank  Neurotrophin signaling pathway Homo sapiens hs...   0.40285   
5   prerank       Small cell lung cancer Homo sapiens hsa05222  0.417544   
6   prerank               Thyroid cancer Homo sapiens hsa05216  0.504779   
7   prerank           Insulin resistance Homo sapiens hsa04931  0.390886   
8   prerank  B cell receptor signaling pathway Homo sapiens...  0.396887   
9   prerank    Insulin signaling pathway Homo sapiens hsa04910  0.378856   
10  prerank       FoxO signaling pathway Homo sapiens hsa04068  0.382864   
11  prerank              Prostate cancer Homo sapiens hsa05215  0.397292   
12  prerank 

In [19]:
print(enumerate(CP_embedding_dict))